# 03 · Experiments E0–E6 (M4, M5)
Each experiment writes `runs/<experiment>__<variant>/` on Drive (config, metrics, per-step CSV, verdicts, traces).
Thresholds are tuned on **dev only**; test is reported once per configuration.
After E2 and E3, pass the winners to later experiments with `--set` (shown below).
**Paste back** the tables printed at the end (or the `exports/RESULTS_tables.md` file).

## Setup
Mounts Drive, clones or updates the repo, installs it and sets the `SOPVERIFY_*` paths. Safe to re-run after a runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%%bash
set -e
REPO=/content/SOP-Verification
BRANCH=claude/nifty-mccarthy-5suut3   # switch to main once the branch is merged
if [ -d $REPO/.git ]; then
  git -C $REPO fetch -q origin $BRANCH && git -C $REPO checkout -q $BRANCH && git -C $REPO pull -q --ff-only origin $BRANCH
else
  git clone -q -b $BRANCH https://github.com/RohanSada/SOP-Verification.git $REPO
fi
pip install -q -e "$REPO[app,dev]"
git -C $REPO log --oneline -1

In [ ]:
import os, sys
sys.path.insert(0, "/content/SOP-Verification/src")   # editable install without a runtime restart
os.chdir("/content/SOP-Verification")
from sopverify.colab import setup_env, show
ROOT = "/content/drive/MyDrive/sopverify"
ENV = setup_env(ROOT)
try:  # optional Hugging Face token (Colab secret named HF_TOKEN), in case the dataset is gated
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
ENV

### E0 oracle (must be perfect) and E1 time prior (no GPU needed)

In [ ]:
!sopverify eval -c configs/experiments/E0_oracle.yaml
!sopverify eval -c configs/experiments/E1_time_prior.yaml

### E2 features (linear probe)

In [ ]:
!sopverify eval -c configs/experiments/E2_features.yaml

### E3 scorers on the E2 winner
The winner is picked on the **dev** split only (targets first, then not-done recall, then F1).

In [ ]:
from sopverify.eval.tables import pick_best
RUNS = os.environ["SOPVERIFY_RUNS"]
b2 = pick_best(RUNS, "E2_features"); print("E2 winner:", b2["variant"], b2["dev"])
f = b2["config"]["features"]
BEST_FEATURES = f"--set features.extractor={f['extractor']} --set features.crop={f['crop']}"
print(BEST_FEATURES)
!sopverify eval -c configs/experiments/E3_scorers.yaml $BEST_FEATURES

### E4–E6 on the E2 × E3 winner

In [ ]:
b3 = pick_best(RUNS, "E3_scorers"); print("E3 winner:", b3["variant"], b3["dev"])
BEST = BEST_FEATURES + f" --set scorer.name={b3['config']['scorer']['name']}"
print(BEST)
!sopverify eval -c configs/experiments/E4_num_demos.yaml $BEST
!sopverify eval -c configs/experiments/E5_granularity.yaml $BEST
!sopverify eval -c configs/experiments/E6_graph.yaml $BEST

### Tables 📋

In [ ]:
!sopverify tables --out "$SOPVERIFY_EXPORTS/RESULTS_tables.md"
from IPython.display import Markdown, display
display(Markdown(open(os.environ["SOPVERIFY_EXPORTS"] + "/RESULTS_tables.md").read()))

### Save the best configuration 📋
The E3 winner's full config (tuned thresholds included) becomes the candidate `configs/best.yaml`. Send me `exports/best.yaml` together with the tables.

In [ ]:
import shutil
shutil.copy(b3["config_path"], f"{os.environ['SOPVERIFY_EXPORTS']}/best.yaml")
print(open(b3["config_path"]).read())